# CLV 조건부 상품군·가격 구매이력 표현 M2 — Dunnhumby seed 42

학습기간 행동으로 계산한 N/V 기반 historical CLV proxy가 고객별 상품군 관계와 상품군 내 가격관계의 사용비율을 결정하는 M2 역사적 개발 실험입니다.

- 학습: Dunnhumby DAY 1~683
- 평가: DAY 684~690의 신규상품
- 사용자 상태: N 수준, V 수준, N×V, N−V
- N: 관찰기간 보정 반복거래율·최근 지속성·거래간격의 행동변수 결합
- V: 거래당 평균 구매금액·거래별 상품군 내 가격위치의 행동변수 결합
- 표현: LightGCN ID 64차원 + 상품군 4차원 + 상품군 내 가격관계 4차원
- N/V는 독립 점수를 만들지 않고 10개 파라미터의 softmax 혼합비만 결정
- 상품 인기도·구매고객 수·RepeatShare 입력 없음
- 학습 positive 상품은 사용자 이력에서 제외하고 2층 전파 결과까지 정확히 보정
- 고정: binary graph, uniform negative sampling, plain BPR, 100 epoch, rho=0.1
- 비교: 같은 역사적 개발 protocol의 저장된 M1@64 결과 재사용

최종 test와 holdout은 만들지 않는 seed 42 탐색이며 유의성·일반화를 주장하지 않습니다.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

REVIEWED_SHA = '09dc87d4a46603327b4cad789a52f9e59d03c89b'
%cd /content
!rm -rf /content/clv-m2-lightgcn-runner
!git clone -q https://github.com/jung-un/clv-m2-lightgcn-runner.git /content/clv-m2-lightgcn-runner
%cd /content/clv-m2-lightgcn-runner
!git checkout -q $REVIEWED_SHA
import subprocess
assert subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip() == REVIEWED_SHA
print('코드 고정 완료:', REVIEWED_SHA)


In [ ]:
import json
import torch
from lightgcn_clv_conditioned_category_price_history import (
    configure_conditioned_history_run,
    preflight_summary,
    run_conditioned_history_screen,
)

assert torch.cuda.is_available(), '런타임 유형에서 GPU를 선택하세요.'
cfg = configure_conditioned_history_run(
    out_dir=(
        '/content/drive/MyDrive/논문/data/'
        'results_v3_dunnhumby_m2_clv_conditioned_category_price_history_historical_screen_v1'
    ),
    baseline_result_dir=(
        '/content/drive/MyDrive/논문/data/'
        'results_v3_dunnhumby_m2_repeatshare_historical_backtest_v1'
    ),
)
summary = preflight_summary(cfg)
assert summary['historical_development_split']['final_test_constructed'] is False
assert summary['historical_development_split']['holdout_constructed'] is False
assert summary['m2']['rho'] == 0.1
assert summary['m2']['positive_item_leave_one_out'].startswith('exact')
assert summary['m2']['raw_item_popularity_input'] is False
assert summary['m2']['raw_repeatshare_input'] is False
assert summary['fixed']['graph'] == 'binary'
assert summary['fixed']['negative_sampling'] == 'uniform'
assert summary['fixed']['sample_weighting'] is False
print(json.dumps(summary, ensure_ascii=False, indent=2))


In [ ]:
result_df = run_conditioned_history_screen(cfg)


In [ ]:
from IPython.display import display

comparison = result_df.attrs['comparison'].copy()
id_only_comparison = result_df.attrs['id_only_comparison'].copy()
reading = dict(result_df.attrs['screening_reading'])
paths = dict(result_df.attrs['result_paths'])
display_df = result_df.copy()
display_df.attrs = {}

core_metrics = [
    'recall@10', 'ndcg@10', 'recall@20', 'ndcg@20',
    'recall@50', 'ndcg@50',
    'price_purchase_amount_weighted_hit@10',
    'price_purchase_amount_weighted_hit@20',
    'price_purchase_amount_weighted_hit@50',
    'coverage@10', 'n_distinct@10', 'top10_share@10',
]
print('절대지표:')
display(display_df.sort_values('model_id'))
print('M1@64 대비 핵심 변화:')
display(comparison[comparison['metric'].isin(core_metrics)].sort_values('metric'))
print('공동학습 ID-only 대비 full 핵심 변화:')
display(id_only_comparison[id_only_comparison['metric'].isin(core_metrics)].sort_values('metric'))
print('탐색 판독:', reading)
print('결과 파일:', paths)
